# pypgo.implicit API Demo

This notebook introduces the lazy implicit surface API:
`GridSpec`, analytic fields, CSG operators, grid sampling,
marching-cubes extraction, and mesh surface thickening.


In [1]:
import numpy as np
import pypgo as pgo
from pypgo import implicit, vis

# PyVista notebook views are interactive by default. For lightweight
# static outputs, uncomment:
# vis.set_backend("static")


## 1. Analytic fields

`SphereField` and `BoxField` are `ImplicitField` objects. They can be
queried at points without allocating a sampled grid.


In [2]:
sphere = implicit.SphereField([0.0, 0.0, 0.0], 1.0)
box = implicit.BoxField([0.0, 0.0, 0.0], [0.75, 0.75, 0.75])

points = np.array(
    [
        [0.0, 0.0, 0.0],
        [1.0, 0.0, 0.0],
        [1.5, 0.0, 0.0],
    ],
    dtype=np.float64,
)

print("sphere eval:", [sphere.eval(p) for p in points])
print("box eval:", [box.eval(p) for p in points])
print("sphere bounds:", sphere.bounds())

sphere_surface = implicit.extract_marching_cubes(
    sphere.sample_to_grid(implicit.GridSpec([-1.2, -1.2, -1.2], [1.2, 1.2, 1.2], resolution=36)),
    iso_offset=0.0,
)
box_surface = implicit.extract_marching_cubes(
    box.sample_to_grid(implicit.GridSpec([-1.0, -1.0, -1.0], [1.0, 1.0, 1.0], resolution=36)),
    iso_offset=0.0,
)
vis.plot_surface(
    [sphere_surface, box_surface],
    titles=["SphereField", "BoxField"],
    show_edges=False,
    colors=["lightsteelblue", "wheat"],
)


sphere eval: [-1.0, 0.0, 0.5]
box eval: [-0.75, 0.25, 0.75]
sphere bounds: (array([-1., -1., -1.]), array([1., 1., 1.]))


Widget(value='<iframe src="http://localhost:52191/index.html?ui=P_0x177a77170_0&reconnect=auto" class="pyvista…

## 2. Lazy CSG

The `|`, `&`, and `-` operators create lazy composed fields. No dense
grid is allocated until `sample_to_grid` is called.


In [9]:
left = implicit.SphereField([-0.45, 0.0, 0.0], 0.9)
right = implicit.SphereField([0.45, 0.0, 0.0], 0.9)

union = left | right
intersection = left & right
difference_left = left - right
difference_right = right - left

p = np.array([0.0, 0.0, 0.0], dtype=np.float64)
print("union at origin:", union.eval(p))
print("intersection at origin:", intersection.eval(p))
print("difference_left at origin:", difference_left.eval(p))
print("difference_right at origin:", difference_right.eval(p))
print("lazy type:", type(union).__name__)

csg_spec = implicit.GridSpec([-1.4, -1.1, -1.1], [1.4, 1.1, 1.1], resolution=100)
csg_surfaces = [
    implicit.extract_marching_cubes(field.sample_to_grid(csg_spec), iso_offset=0.0)
    for field in (union, intersection, difference_left, difference_right)
]
vis.plot_surface(
    csg_surfaces,
    titles=["union", "intersection", "difference_left", "difference_right"],
    show_edges=False,
    colors=["lightsteelblue", "thistle", "salmon", "lightcoral"],
    window_size=(960, 320),
)


union at origin: -0.45
intersection at origin: -0.45
difference_left at origin: 0.45
difference_right at origin: 0.45
lazy type: ImplicitField


Widget(value='<iframe src="http://localhost:52191/index.html?ui=P_0x319a93c20_7&reconnect=auto" class="pyvista…

## 3. Sampling to `GridField`

`sample_to_grid` materializes a field on a uniform grid. `GridField`
still inherits `ImplicitField`, so point queries use trilinear
interpolation and CSG operators continue to work. Sampling uses
`pgo.parallel` defaults unless a call passes `num_threads`; `None`
means automatic backend defaults, and `1` is useful for serial debug runs.


In [5]:
spec = implicit.GridSpec([-1.5, -1.5, -1.5], [1.5, 1.5, 1.5], resolution=48)

print("default libpgo workers:", pgo.parallel.get_num_threads())
with pgo.parallel.thread_limit(1):
    serial_grid = union.sample_to_grid(spec)
    print("scoped serial grid shape:", serial_grid.values.shape)

pgo.parallel.set_num_threads(4)
grid = union.sample_to_grid(spec)  # uses the global libpgo default
pgo.parallel.set_num_threads(None)

values = grid.values
print("grid shape:", values.shape)
print("grid dtype:", values.dtype)
print("min/max:", float(values.min()), float(values.max()))
print("grid eval at origin:", grid.eval([0.0, 0.0, 0.0]))
print("values shares memory:", np.shares_memory(values, grid.values))

grid_preview = implicit.extract_marching_cubes(grid, iso_offset=0.0)
vis.plot_surface(grid_preview, titles=["sampled GridField iso-surface"], show_edges=False)


default libpgo workers: None
scoped serial grid shape: (48, 48, 48)
grid shape: (48, 48, 48)
grid dtype: float64
min/max: -0.8465008895290207 1.4669600757089252
grid eval at origin: -0.4794856993532347
values shares memory: True


Widget(value='<iframe src="http://localhost:52191/index.html?ui=P_0x319b8de20_3&reconnect=auto" class="pyvista…

## 4. Marching cubes

Extraction accepts a materialized `GridField`. For an unsampled lazy
field, call `sample_to_grid` first.


In [6]:
surface = implicit.extract_marching_cubes(grid, iso_offset=0.0)
print("vertices:", surface.num_vertices)
print("triangles:", surface.num_elements)
print("bbox:", surface.bbox)

vis.plot_surface(surface, titles=["lazy CSG union"], show_edges=False, colors=["lightsteelblue"])


vertices: 5528
triangles: 11052
bbox: (array([-1.3488662 , -0.89897393, -0.89897393]), array([1.3488662 , 0.89897393, 0.89897393]))


Widget(value='<iframe src="http://localhost:52191/index.html?ui=P_0x319a93410_4&reconnect=auto" class="pyvista…

## 5. Mesh unsigned distance and shell thickening

`MeshUnsignedDistanceField` turns a triangle surface into a distance
field. The convenience function below performs the common pipeline:
mesh -> distance field -> offset -> grid -> marching cubes.


In [7]:
tri = pgo.mesh.TriMeshData(
    [[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0]],
    [[0, 1, 2]],
)

shell = implicit.thicken_mesh_surface(
    tri,
    thickness=0.1,
    resolution=24,
    padding=0.25,
)

print("shell vertices:", shell.num_vertices)
print("shell triangles:", shell.num_elements)

vis.plot_surface([tri, shell], titles=["input triangle", "thickened shell"], colors=["lightgray", "salmon"])


shell vertices: 558
shell triangles: 1112


Widget(value='<iframe src="http://localhost:52191/index.html?ui=P_0x31e131190_5&reconnect=auto" class="pyvista…

## 6. OpenVDB availability

OpenVDB support depends on how libpgo was built. Use `has_openvdb`
before constructing OpenVDB level sets.


In [8]:
print("OpenVDB available:", implicit.has_openvdb())

if implicit.has_openvdb():
    opts = implicit.OpenVDBOptions(voxel_size=0.05)
    levelset = implicit.build_openvdb_from_grid_field(grid, opts)
    vdb_surface = implicit.extract_openvdb(levelset, opts)
    print("OpenVDB surface:", vdb_surface.num_vertices, vdb_surface.num_elements)
    vis.plot_surface(vdb_surface, titles=["OpenVDB extraction"], show_edges=False, colors=["palegreen"])


OpenVDB available: True
OpenVDB surface: 5530 11056


Widget(value='<iframe src="http://localhost:52191/index.html?ui=P_0x319b8f380_6&reconnect=auto" class="pyvista…